In [ ]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 8.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# MANUAL TEST OF CHARACTER MODEL
# INDIAN NUMBER PLATE ORDER + VALIDATION
# ============================================================

from google.colab import drive, files
from ultralytics import YOLO
import torch
import matplotlib.pyplot as plt
import os
import re


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 2. MODEL PATH
# ============================================================

MODEL_PATH = (
    "/content/drive/MyDrive/"
    "AI_Training/Models/"
    "char_epoch2/best.pt"
)


if not os.path.isfile(MODEL_PATH):

    raise FileNotFoundError(
        f"Model not found:\n{MODEL_PATH}"
    )


print("=" * 70)
print("INDIAN NUMBER PLATE CHARACTER MODEL TEST")
print("=" * 70)

print("\nModel:")
print(MODEL_PATH)


# ============================================================
# 3. LOAD MODEL
# ============================================================

model = YOLO(MODEL_PATH)

print("\n✓ Model loaded successfully.")


# ============================================================
# 4. DEVICE
# ============================================================

if torch.cuda.is_available():

    DEVICE = 0

    print(
        "✓ GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    DEVICE = "cpu"

    print(
        "⚠ GPU not available — using CPU."
    )


# ============================================================
# 5. INDIAN STATE / UT CODES
# ============================================================

STATE_CODES = {

    "AN": "Andaman and Nicobar Islands",
    "AP": "Andhra Pradesh",
    "AR": "Arunachal Pradesh",
    "AS": "Assam",
    "BR": "Bihar",
    "CH": "Chandigarh",
    "CG": "Chhattisgarh",
    "DD": "Dadra and Nagar Haveli and Daman and Diu",
    "DN": "Dadra and Nagar Haveli and Daman and Diu",
    "DL": "Delhi",
    "GA": "Goa",
    "GJ": "Gujarat",
    "HR": "Haryana",
    "HP": "Himachal Pradesh",
    "JK": "Jammu and Kashmir",
    "JH": "Jharkhand",
    "KA": "Karnataka",
    "KL": "Kerala",
    "LA": "Ladakh",
    "LD": "Lakshadweep",
    "MP": "Madhya Pradesh",
    "MH": "Maharashtra",
    "MN": "Manipur",
    "ML": "Meghalaya",
    "MZ": "Mizoram",
    "NL": "Nagaland",
    "OD": "Odisha",
    "PB": "Punjab",
    "PY": "Puducherry",
    "RJ": "Rajasthan",
    "SK": "Sikkim",
    "TN": "Tamil Nadu",
    "TS": "Telangana",
    "TR": "Tripura",
    "UP": "Uttar Pradesh",
    "UK": "Uttarakhand",
    "WB": "West Bengal"
}


# ============================================================
# 6. OCR CONFUSION MAP
# ============================================================

# Used ONLY when a position is expected to contain a digit.

LETTER_TO_DIGIT = {

    "O": "0",
    "D": "0",
    "Q": "0",

    "I": "1",
    "L": "1",

    "Z": "2",

    "S": "5",

    "G": "6",

    "T": "7",

    "B": "8"
}


# Used ONLY when a position is expected to contain a letter.

DIGIT_TO_LETTER = {

    "0": "O",
    "1": "I",
    "2": "Z",
    "5": "S",
    "6": "G",
    "8": "B"
}


# ============================================================
# 7. SELECT IMAGE
# ============================================================

print("\n" + "=" * 70)
print("SELECT YOUR IMAGE")
print("=" * 70)

uploaded = files.upload()


if not uploaded:

    raise RuntimeError(
        "No image was selected."
    )


IMAGE_PATH = next(iter(uploaded))

print("\n✓ Selected image:")
print(IMAGE_PATH)


# ============================================================
# 8. RUN CHARACTER DETECTION
# ============================================================

print("\nRunning character detection...")

results = model.predict(

    source=IMAGE_PATH,

    imgsz=640,

    conf=0.30,

    iou=0.45,

    device=DEVICE,

    verbose=False
)


result = results[0]

print("✓ Detection completed.")


# ============================================================
# 9. DISPLAY RESULT
# ============================================================

annotated = result.plot()


plt.figure(
    figsize=(16, 8)
)


plt.imshow(
    annotated[:, :, ::-1]
)


plt.axis("off")


plt.title(
    "Character Detection Result"
)


plt.show()


# ============================================================
# 10. EXTRACT DETECTIONS
# ============================================================

print("\n" + "=" * 70)
print("DETECTED CHARACTERS")
print("=" * 70)


if result.boxes is None or len(result.boxes) == 0:

    print("\n❌ No characters detected.")

else:

    detections = []


    for box in result.boxes:

        xyxy = (
            box.xyxy[0]
            .cpu()
            .numpy()
        )


        x1 = float(xyxy[0])
        y1 = float(xyxy[1])
        x2 = float(xyxy[2])
        y2 = float(xyxy[3])


        confidence = float(
            box.conf[0]
            .cpu()
            .item()
        )


        class_id = int(
            box.cls[0]
            .cpu()
            .item()
        )


        character = str(
            model.names[class_id]
        ).upper()


        center_x = (
            x1 + x2
        ) / 2


        center_y = (
            y1 + y2
        ) / 2


        detections.append({

            "x": center_x,

            "y": center_y,

            "x1": x1,

            "y1": y1,

            "x2": x2,

            "y2": y2,

            "character": character,

            "confidence": confidence

        })


    # ========================================================
    # 11. SORT LEFT → RIGHT
    # ========================================================

    detections.sort(
        key=lambda d: d["x"]
    )


    # ========================================================
    # 12. PRINT DETECTIONS
    # ========================================================

    for i, d in enumerate(
        detections,
        1
    ):

        print(

            f"{i:2d}. "

            f"{d['character']}  "

            f"confidence = "
            f"{d['confidence']:.3f}  "

            f"x = "
            f"{d['x']:.1f}"

        )


    # ========================================================
    # 13. RAW SEQUENCE
    # ========================================================

    raw_sequence = "".join(

        d["character"]

        for d in detections

    )


    print("\n" + "-" * 70)

    print(
        "RAW SORTED SEQUENCE"
    )

    print("-" * 70)

    print(
        raw_sequence
    )


    # ========================================================
    # 14. IDENTIFY STATE CODE
    # ========================================================

    state_code = raw_sequence[:2]


    print("\n" + "-" * 70)

    print(
        "STATE CODE CHECK"
    )

    print("-" * 70)


    if state_code in STATE_CODES:

        print(
            f"✓ State Code : {state_code}"
        )

        print(
            f"✓ State      : "
            f"{STATE_CODES[state_code]}"
        )

    else:

        print(
            f"⚠ Unknown state code: "
            f"{state_code}"
        )


    # ========================================================
    # 15. POSITION-BASED CORRECTION
    # ========================================================

    chars = list(raw_sequence)


    # --------------------------------------------------------
    # NORMAL STATE PLATE
    #
    # First 2 positions = letters
    # Following positions are interpreted based on structure.
    # --------------------------------------------------------

    if len(chars) >= 4:

        # ----------------------------------------------------
        # First two must be alphabetic state code
        # ----------------------------------------------------

        for i in range(
            min(2, len(chars))
        ):

            if chars[i].isdigit():

                if chars[i] in DIGIT_TO_LETTER:

                    chars[i] = DIGIT_TO_LETTER[
                        chars[i]
                    ]


    # ========================================================
    # 16. FIND STATE AFTER POSSIBLE CORRECTION
    # ========================================================

    corrected_state = (
        "".join(chars[:2])
    )


    # ========================================================
    # 17. NORMAL PLATE PATTERN CHECK
    # ========================================================

    def is_normal_plate(s):

        patterns = [

            # XX1A1234
            r"^[A-Z]{2}[0-9]{1}[A-Z]{1}[0-9]{4}$",

            # XX01A1234
            r"^[A-Z]{2}[0-9]{2}[A-Z]{1}[0-9]{4}$",

            # XX1AB1234
            r"^[A-Z]{2}[0-9]{1}[A-Z]{2}[0-9]{4}$",

            # XX01AB1234
            r"^[A-Z]{2}[0-9]{2}[A-Z]{2}[0-9]{4}$",

            # XX01ABC1234
            r"^[A-Z]{2}[0-9]{2}[A-Z]{3}[0-9]{4}$"
        ]


        return any(
            re.fullmatch(
                pattern,
                s
            )
            for pattern in patterns
        )


    # ========================================================
    # 18. BH SERIES CHECK
    # ========================================================

    def is_bh_plate(s):

        # YYBH####AA

        pattern = (
            r"^[0-9]{2}"
            r"BH"
            r"[0-9]{4}"
            r"[A-Z]{2}$"
        )

        return bool(
            re.fullmatch(
                pattern,
                s
            )
        )


    # ========================================================
    # 19. POSITION-AWARE CORRECTION
    # ========================================================

    def correct_normal_plate(s):

        chars = list(s)


        # ----------------------------------------------------
        # STATE
        # ----------------------------------------------------

        for i in range(
            min(2, len(chars))
        ):

            if chars[i].isdigit():

                if chars[i] in DIGIT_TO_LETTER:

                    chars[i] = (
                        DIGIT_TO_LETTER[
                            chars[i]
                        ]
                    )


        # ----------------------------------------------------
        # Try possible normal structures
        # ----------------------------------------------------

        possible_patterns = [

            # state + 1 digit + 1 letter + 4 digits
            [2, 1, 1, 4],

            # state + 2 digits + 1 letter + 4 digits
            [2, 2, 1, 4],

            # state + 1 digit + 2 letters + 4 digits
            [2, 1, 2, 4],

            # state + 2 digits + 2 letters + 4 digits
            [2, 2, 2, 4],

            # state + 2 digits + 3 letters + 4 digits
            [2, 2, 3, 4]
        ]


        # ----------------------------------------------------
        # Choose pattern matching total length
        # ----------------------------------------------------

        selected = None


        for pattern in possible_patterns:

            if sum(pattern) == len(chars):

                selected = pattern

                break


        if selected is None:

            return "".join(chars)


        # ----------------------------------------------------
        # Process sections
        # ----------------------------------------------------

        position = 0


        # STATE
        state_length = selected[0]

        position += state_length


        # RTO
        rto_length = selected[1]

        for i in range(
            position,
            position + rto_length
        ):

            c = chars[i]

            if c.isalpha():

                if c in LETTER_TO_DIGIT:

                    chars[i] = (
                        LETTER_TO_DIGIT[c]
                    )


        position += rto_length


        # SERIES
        series_length = selected[2]

        for i in range(
            position,
            position + series_length
        ):

            c = chars[i]

            if c.isdigit():

                if c in DIGIT_TO_LETTER:

                    chars[i] = (
                        DIGIT_TO_LETTER[c]
                    )


        position += series_length


        # SERIAL NUMBER
        number_length = selected[3]

        for i in range(
            position,
            position + number_length
        ):

            c = chars[i]

            if c.isalpha():

                if c in LETTER_TO_DIGIT:

                    chars[i] = (
                        LETTER_TO_DIGIT[c]
                    )


        return "".join(chars)


    # ========================================================
    # 20. APPLY CORRECTION
    # ========================================================

    corrected_sequence = (
        correct_normal_plate(
            raw_sequence
        )
    )


    # ========================================================
    # 21. CHECK BH SERIES
    # ========================================================

    if is_bh_plate(
        corrected_sequence
    ):

        plate_type = (
            "BH SERIES"
        )

        final_sequence = (
            corrected_sequence
        )


    elif is_normal_plate(
        corrected_sequence
    ):

        plate_type = (
            "NORMAL STATE REGISTRATION"
        )

        final_sequence = (
            corrected_sequence
        )


    else:

        plate_type = (
            "UNKNOWN / SPECIAL FORMAT"
        )

        final_sequence = (
            corrected_sequence
        )


    # ========================================================
    # 22. FINAL RESULT
    # ========================================================

    print("\n" + "=" * 70)

    print(
        "FINAL NUMBER PLATE"
    )

    print("=" * 70)


    print(
        f"Raw sequence       : "
        f"{raw_sequence}"
    )


    print(
        f"Corrected sequence : "
        f"{corrected_sequence}"
    )


    print(
        f"Plate type         : "
        f"{plate_type}"
    )


    # ========================================================
    # 23. FINAL STATE
    # ========================================================

    final_state = (
        corrected_sequence[:2]
    )


    if final_state in STATE_CODES:

        print(
            f"State              : "
            f"{STATE_CODES[final_state]}"
        )

        print(
            f"State code         : "
            f"{final_state}"
        )


    print("=" * 70)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Mounted at /content/drive
INDIAN NUMBER PLATE CHARACTER MODEL TEST

Model:
/content/drive/MyDrive/AI_Training/Models/char_epoch2/best.pt

✓ Model loaded successfully.
✓ GPU: Tesla T4

SELECT YOUR IMAGE
